# Debug Model: Logit Analysis of Correct vs Misclassified Samples

Run the trained FIDS30 classifier on the test set and compare the raw logit distributions for correctly classified and misclassified samples.

In [ ]:
import torch
import torch.nn.functional as F
import timm
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets
from torch.utils.data import DataLoader

NUM_CLASSES = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)
model.load_state_dict(torch.load("fids30_classifier_30cls_b0.pth", map_location=device))
model = model.to(device)
model.eval()

config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**config, is_training=False)
test_ds = datasets.ImageFolder("PrepData/Test", transform=transform)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

class_names = test_ds.classes
print(f"Device: {device}")
print(f"Test samples: {len(test_ds)}")
print(f"Classes ({len(class_names)}): {class_names}")

## 1. Collect Logits for All Test Samples

In [ ]:
all_logits = []
all_labels = []
all_preds = []
all_images = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1)

        all_logits.append(logits.cpu())
        all_labels.append(labels)
        all_preds.append(preds.cpu())
        all_images.append(images.cpu())

all_logits = torch.cat(all_logits)
all_labels = torch.cat(all_labels)
all_preds = torch.cat(all_preds)
all_images = torch.cat(all_images)
all_probs = F.softmax(all_logits, dim=1)

correct_mask = all_preds == all_labels
n_correct = correct_mask.sum().item()
n_wrong = (~correct_mask).sum().item()
print(f"Correct: {n_correct}, Misclassified: {n_wrong} ({100*n_wrong/len(all_labels):.1f}%)")

## 2. Confidence Distribution: Correct vs Misclassified

Compare the predicted-class probability (confidence) for correctly vs incorrectly classified samples.

In [ ]:
correct_conf = all_probs[correct_mask].max(dim=1).values.numpy()
wrong_conf = all_probs[~correct_mask].max(dim=1).values.numpy()

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 1, 30)
ax.hist(correct_conf, bins=bins, alpha=0.7, label=f"Correct (n={n_correct})", color="steelblue")
if n_wrong > 0:
    ax.hist(wrong_conf, bins=bins, alpha=0.7, label=f"Misclassified (n={n_wrong})", color="tomato")
ax.set_xlabel("Predicted-Class Probability (Confidence)")
ax.set_ylabel("Count")
ax.set_title("Confidence Distribution: Correct vs Misclassified")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Top-5 Logit & Probability Bar Charts: Misclassified Samples

Show the image alongside the top-5 logits and probabilities for each misclassified sample.

In [ ]:
def denorm_image(img_tensor):
    """Convert normalized tensor back to displayable [0,1] RGB image."""
    img = img_tensor.numpy().transpose(1, 2, 0)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img

def plot_top5_bars(indices, title, max_show=8):
    """Plot image + top-5 logit and probability bar charts for given sample indices."""
    indices = indices[:max_show]
    n = len(indices)
    if n == 0:
        print(f"No samples to show for: {title}")
        return

    fig, axes = plt.subplots(n, 3, figsize=(16, 2.5 * n),
                             gridspec_kw={"width_ratios": [1, 2, 2]})
    fig.suptitle(title, fontsize=14, y=1.01)
    if n == 1:
        axes = axes.reshape(1, 3)

    for row, idx in enumerate(indices):
        logits = all_logits[idx].numpy()
        probs = all_probs[idx].numpy()
        true_label = all_labels[idx].item()
        pred_label = all_preds[idx].item()

        # Top-5 indices (by logit value)
        top5_idx = np.argsort(logits)[::-1][:5]
        top5_names = [class_names[i] for i in top5_idx]
        top5_logits = logits[top5_idx]
        top5_probs = probs[top5_idx]

        # Colors: blue for true class, red for predicted, gray for others
        def get_colors(indices):
            colors = []
            for i in indices:
                if i == true_label and i == pred_label:
                    colors.append('steelblue')
                elif i == pred_label:
                    colors.append('tomato')
                elif i == true_label:
                    colors.append('steelblue')
                else:
                    colors.append('lightgray')
            return colors

        colors = get_colors(top5_idx)

        # Image
        axes[row, 0].imshow(denorm_image(all_images[idx]))
        axes[row, 0].set_title(f"True: {class_names[true_label]}", fontsize=10)
        axes[row, 0].axis('off')

        # Top-5 logit bar chart
        bars_l = axes[row, 1].barh(top5_names[::-1], top5_logits[::-1], color=colors[::-1])
        for bar, val in zip(bars_l, top5_logits[::-1]):
            axes[row, 1].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                              f"{val:.2f}", va='center', fontsize=8)
        axes[row, 1].set_xlabel("Logit")
        axes[row, 1].set_title(f"Top-5 Logits (pred: {class_names[pred_label]})", fontsize=10)

        # Top-5 probability bar chart
        bars_p = axes[row, 2].barh(top5_names[::-1], top5_probs[::-1], color=colors[::-1])
        for bar, val in zip(bars_p, top5_probs[::-1]):
            axes[row, 2].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                              f"{val:.2%}", va='center', fontsize=8)
        axes[row, 2].set_xlabel("Probability")
        axes[row, 2].set_xlim(0, 1)
        axes[row, 2].set_title(f"Top-5 Probabilities", fontsize=10)

    from matplotlib.patches import Patch
    legend_elements = [Patch(color='steelblue', label='True class'),
                       Patch(color='tomato', label='Predicted class'),
                       Patch(color='lightgray', label='Other')]
    fig.legend(handles=legend_elements, loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()

# Show misclassified samples
wrong_indices = torch.where(~correct_mask)[0].tolist()
plot_top5_bars(wrong_indices, "Misclassified Samples — Top-5 Logits & Probabilities")

## 4. Top-5 Logit & Probability Bar Charts: Correctly Classified Samples

Show a random subset of correct predictions for comparison — these should have a dominant top-1 logit/probability.

In [ ]:
# Pick one random correct sample per class for variety
rng = np.random.default_rng(42)
correct_indices = []
for cls_idx in range(len(class_names)):
    cls_correct = torch.where(correct_mask & (all_labels == cls_idx))[0].tolist()
    if cls_correct:
        correct_indices.append(rng.choice(cls_correct))

plot_top5_bars(correct_indices, "Correctly Classified Samples — Top-5 Logits & Probabilities (1 per class)")

## 5. Saliency Maps

Compute the gradient of the predicted class score w.r.t. the input pixels. Bright regions indicate pixels that most influence the prediction.

In [ ]:
def compute_saliency(model, image_tensor, target_class):
    """Compute saliency map for a single image w.r.t. target class."""
    image = image_tensor.unsqueeze(0).to(device).requires_grad_(True)
    output = model(image)
    output[0, target_class].backward()
    saliency = image.grad.data.abs().squeeze(0).max(dim=0).values  # max across RGB channels
    return saliency.cpu().numpy()

def plot_saliency_maps(indices, title, use_true_class=True, max_show=8):
    """Plot image, saliency map, and overlay for given sample indices."""
    indices = indices[:max_show]
    n = len(indices)
    if n == 0:
        print(f"No samples to show for: {title}")
        return

    fig, axes = plt.subplots(n, 3, figsize=(12, 3 * n))
    fig.suptitle(title, fontsize=14, y=1.01)
    if n == 1:
        axes = axes.reshape(1, 3)

    for row, idx in enumerate(indices):
        true_label = all_labels[idx].item()
        pred_label = all_preds[idx].item()
        target = true_label if use_true_class else pred_label

        img_display = denorm_image(all_images[idx])
        saliency = compute_saliency(model, all_images[idx], target)

        # Original image
        axes[row, 0].imshow(img_display)
        axes[row, 0].set_title(f"True: {class_names[true_label]}", fontsize=10)
        axes[row, 0].axis('off')

        # Saliency map
        axes[row, 1].imshow(saliency, cmap='hot')
        axes[row, 1].set_title(f"Saliency (target: {class_names[target]})", fontsize=10)
        axes[row, 1].axis('off')

        # Overlay
        axes[row, 2].imshow(img_display)
        axes[row, 2].imshow(saliency, cmap='hot', alpha=0.5)
        axes[row, 2].set_title(f"Overlay (pred: {class_names[pred_label]})", fontsize=10)
        axes[row, 2].axis('off')

    plt.tight_layout()
    plt.show()

# Misclassified samples — saliency w.r.t. predicted class
wrong_indices = torch.where(~correct_mask)[0].tolist()
plot_saliency_maps(wrong_indices, "Saliency Maps — Misclassified Samples (target = predicted class)", use_true_class=False)

# Correctly classified samples — one per class
rng = np.random.default_rng(42)
correct_indices = []
for cls_idx in range(len(class_names)):
    cls_correct = torch.where(correct_mask & (all_labels == cls_idx))[0].tolist()
    if cls_correct:
        correct_indices.append(rng.choice(cls_correct))

plot_saliency_maps(correct_indices, "Saliency Maps — Correctly Classified Samples (target = true class)", use_true_class=True)

## 6. Occlusion Sensitivity

Slide a gray patch across the image and record how the predicted-class probability changes.
Regions where occlusion causes a large drop are critical for the prediction.

In [ ]:
def occlusion_sensitivity(model, img_tensor, target_class, patch_size=16, stride=4):
    """Compute occlusion sensitivity map by sliding a gray patch over the image."""
    img = img_tensor.unsqueeze(0).to(device)
    _, C, H, W = img.shape

    # Baseline probability
    with torch.no_grad():
        base_prob = F.softmax(model(img), dim=1)[0, target_class].item()

    # Build all occluded images in one pass for efficiency
    rows = range(0, H - patch_size + 1, stride)
    cols = range(0, W - patch_size + 1, stride)
    sensitivity = np.zeros((len(list(rows)), len(list(cols))))

    rows = list(range(0, H - patch_size + 1, stride))
    cols = list(range(0, W - patch_size + 1, stride))

    # Process in batches to avoid OOM
    batch = []
    positions = []
    for r_i, r in enumerate(rows):
        for c_i, c in enumerate(cols):
            occluded = img.clone()
            occluded[:, :, r:r+patch_size, c:c+patch_size] = 0  # gray patch
            batch.append(occluded)
            positions.append((r_i, c_i))

            if len(batch) == 64:
                with torch.no_grad():
                    probs = F.softmax(model(torch.cat(batch)), dim=1)[:, target_class].cpu().numpy()
                for (ri, ci), p in zip(positions, probs):
                    sensitivity[ri, ci] = base_prob - p
                batch, positions = [], []

    # Remaining
    if batch:
        with torch.no_grad():
            probs = F.softmax(model(torch.cat(batch)), dim=1)[:, target_class].cpu().numpy()
        for (ri, ci), p in zip(positions, probs):
            sensitivity[ri, ci] = base_prob - p

    return sensitivity


def plot_occlusion(indices, title, use_true_class=True, max_show=8, patch_size=16, stride=4):
    """Plot occlusion sensitivity maps."""
    indices = indices[:max_show]
    n = len(indices)
    if n == 0:
        print(f"No samples to show for: {title}")
        return

    fig, axes = plt.subplots(n, 3, figsize=(12, 3.5 * n))
    fig.suptitle(title, fontsize=14, y=1.01)
    if n == 1:
        axes = axes.reshape(1, 3)

    for row, idx in enumerate(indices):
        true_label = all_labels[idx].item()
        pred_label = all_preds[idx].item()
        target = true_label if use_true_class else pred_label

        img_display = denorm_image(all_images[idx])
        sens = occlusion_sensitivity(model, all_images[idx], target,
                                     patch_size=patch_size, stride=stride)

        # Original image
        axes[row, 0].imshow(img_display)
        axes[row, 0].set_title(f"True: {class_names[true_label]}", fontsize=10)
        axes[row, 0].axis('off')

        # Sensitivity heatmap
        im = axes[row, 1].imshow(sens, cmap='RdBu_r', aspect='auto',
                                  vmin=-np.abs(sens).max(), vmax=np.abs(sens).max())
        axes[row, 1].set_title(f"Occlusion map (target: {class_names[target]})", fontsize=10)
        axes[row, 1].axis('off')
        plt.colorbar(im, ax=axes[row, 1], fraction=0.046, pad=0.04)

        # Overlay: resize sensitivity to image size and overlay
        from PIL import Image
        sens_resized = np.array(Image.fromarray(sens).resize((img_display.shape[1], img_display.shape[0]),
                                                              Image.BILINEAR))
        axes[row, 2].imshow(img_display)
        axes[row, 2].imshow(sens_resized, cmap='RdBu_r', alpha=0.5,
                             vmin=-np.abs(sens).max(), vmax=np.abs(sens).max())
        axes[row, 2].set_title(f"Overlay (pred: {class_names[pred_label]})", fontsize=10)
        axes[row, 2].axis('off')

    plt.tight_layout()
    plt.show()

# Misclassified samples — occlusion w.r.t. predicted class
wrong_indices = torch.where(~correct_mask)[0].tolist()
plot_occlusion(wrong_indices, "Occlusion Sensitivity — Misclassified Samples (target = predicted class)",
               use_true_class=False)

# Correctly classified samples — one per class
rng = np.random.default_rng(42)
correct_indices = []
for cls_idx in range(len(class_names)):
    cls_correct = torch.where(correct_mask & (all_labels == cls_idx))[0].tolist()
    if cls_correct:
        correct_indices.append(rng.choice(cls_correct))

plot_occlusion(correct_indices, "Occlusion Sensitivity — Correctly Classified Samples (target = true class)",
               use_true_class=True)